<a href="https://colab.research.google.com/github/sgulyano/aic403/blob/main/lab1B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIC-403 Lab 1B: MLflow with Databricks

CMKL University

By Sarun Gulyanon


### Goal

Learn how to set up and use MLflow for experiment tracking and model management on Databricks.


### Outline

In this lab, we will walk through an example of using MLflow on Databricks, including logging experiments, tracking metrics, and managing models.

----

# 0. Setting Up Working Environment

- **Sign up for Databricks**: Go to [Databricks](https://www.databricks.com/)
 and create an account if you don’t already have one.
- **Generate an Access Token**:
  - Click on your profile icon in the top-right corner
  - Navigate to Settings > Developer > Access Tokens > Manage.
  - Select `Generate new token` and copy the token for later use.

Finally, install required libraries.

In [1]:
%pip install --upgrade "mlflow-skinny[databricks]"

In [2]:
import os
import mlflow

In [3]:
mlflow.__version__

'3.3.1'

Configure MLflow to log experiment results to the Databricks MLflow Tracking Server

In [4]:
from google.colab import userdata

# MAKE SURE TO KEEP THESE TWO VALUES SECURED
# set to your server URI, copy from URL of your workspace
os.environ["DATABRICKS_HOST"] = userdata.get('DATABRICKS_HOST') #"https://dbc-1234567890123456.cloud.databricks.com"
# use the token we set up earlier,
os.environ["DATABRICKS_TOKEN"] = userdata.get('DATABRICKS_TOKEN') # "dapixxxxxxxxxxxxx"

mlflow.set_tracking_uri("databricks")

# 1. K-Nearest Neighbors (KNN)

K-Nearest Neighbors (KNN) is a simple, non-parametric algorithm that classifies a data point based on the majority label of its `k` closest neighbors in the feature space.

In this lab, we will work with the [Iris dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#iris-dataset), a classic dataset containing measurements of flower sepals and petals from three Iris species. We will use only the first two features (sepal length and sepal width) for visualization purposes.

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [6]:
from sklearn.datasets import load_iris

data = load_iris()
X_clf, y_clf = data.data[:,:2], data.target

Sanity Check: Display some rows using Pandas

In [7]:
# Display as a DataFrame
clf_data = pd.DataFrame(X_clf, columns=["Feature 1", "Feature 2"])
clf_data["Target"] = y_clf
clf_data

,Feature 1,Feature 2,Target
0,5.1,3.5,0
1,4.9,3.0,0
2,4.7,3.2,0
3,4.6,3.1,0
4,5.0,3.6,0
...,...,...,...
145,6.7,3.0,2
146,6.3,2.5,2
147,6.5,3.0,2
148,6.2,3.4,2


Perform a train-test split (hold-out evaluation) to separate training and testing data. Train a KNN classifier with `k=5`.

In [8]:
from sklearn.neighbors import KNeighborsClassifier

# Split data
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)

# Train KNN model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_clf, y_train_clf)

KNeighborsClassifier()

Use the trained model to make predictions. Evaluate model performance using accuracy, precision, recall, and F1-score, which are typical classification metrics.

In [9]:
from sklearn.metrics import classification_report

# Evaluate model
y_pred_clf = knn.predict(X_test_clf)
print(classification_report(y_test_clf, y_pred_clf))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       0.62      0.62      0.62        13
           2       0.62      0.62      0.62        13

    accuracy                           0.78        45
   macro avg       0.74      0.74      0.74        45
weighted avg       0.78      0.78      0.78        45



# 2. MLflow

[MLflow](https://mlflow.org/) is an open-source platform for managing the machine learning lifecycle, including experiment tracking, model management, and deployment.

In [10]:
import mlflow
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

To track your experiments in MLflow, follow these main steps:

1. Set an Experiment. All runs logged under this name will be grouped together. An experiment name must be an absolute path within the Databricks workspace, e.g., `/Users/<some-username>/my-experiment`.

In [11]:
# Create or set experiment
experiment_name = "/knn-optimization"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/3190966787774462', creation_time=1756090095021, experiment_id='3190966787774462', last_update_time=1756090156793, lifecycle_stage='active', name='/knn-optimization', tags={'mlflow.databricks.filesystem.experiment_permissions_check': 'test',
 'mlflow.experiment.sourceName': '/knn-optimization',
 'mlflow.experimentKind': 'custom_model_development',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'sarun@cmkl.ac.th',
 'mlflow.ownerId': '76733628323671'}>

2. Start a Run. Use `with mlflow.start_run():` to start logging a specific run.
3. Log Parameters. Use `mlflow.log_param("param_name", value)` and `mlflow.log_metric("metric_name", value)` to log parameters and metrics.

In [12]:
print(f"Starting hyperparameter optimization experiment: {experiment_name}")

for k in range(2,7):
    with mlflow.start_run(run_name=f"KNN_k={k}"):
        # Train model
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_train_clf, y_train_clf)

        # Predict
        y_pred_clf = knn.predict(X_test_clf)

        # Calculate metrics (weighted average for multiclass)
        acc = accuracy_score(y_test_clf, y_pred_clf)
        prec, rec, f1, _ = precision_recall_fscore_support(y_test_clf, y_pred_clf, average="weighted", zero_division=0)

        # Log parameters and metrics
        mlflow.log_param("n_neighbors", k)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1-score", f1)

        # Log the model
        mlflow.sklearn.log_model(knn, name=f"KNNmodel_k={k}", input_example=X_train_clf)

        print(f"k={k} → Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")

Starting hyperparameter optimization experiment: /knn-optimization
k=2 → Acc: 0.7778, Prec: 0.7958, Rec: 0.7778, F1: 0.7653
🏃 View run KNN_k=2 at: https://dbc-63962062-f37e.cloud.databricks.com/ml/experiments/3190966787774462/runs/08cf13e393c54272a58d435c399954f7
🧪 View experiment at: https://dbc-63962062-f37e.cloud.databricks.com/ml/experiments/3190966787774462
k=3 → Acc: 0.7556, Prec: 0.7558, Rec: 0.7556, F1: 0.7552
🏃 View run KNN_k=3 at: https://dbc-63962062-f37e.cloud.databricks.com/ml/experiments/3190966787774462/runs/47bb3ea865974bd8ab05754eae61202e
🧪 View experiment at: https://dbc-63962062-f37e.cloud.databricks.com/ml/experiments/3190966787774462
k=4 → Acc: 0.7333, Prec: 0.7393, Rec: 0.7333, F1: 0.7183
🏃 View run KNN_k=4 at: https://dbc-63962062-f37e.cloud.databricks.com/ml/experiments/3190966787774462/runs/45050596e5a84b4fa58631c080f88b74
🧪 View experiment at: https://dbc-63962062-f37e.cloud.databricks.com/ml/experiments/3190966787774462
k=5 → Acc: 0.7778, Prec: 0.7778, Rec: 0

View the results in Databricks by navigating to `Experiments` in the left-hand panel.

To manage a model, first identify the best run and register it. Select `Register model` from the run details. Ensure the model name follows the catalog path format, e.g., `workspace.default.<model_name>`. Once registered, you can later load the model and use it for prediction or deployment with the following code:

In [13]:
model_uri = 'models:/workspace.default.knn_iris/1'
model = mlflow.pyfunc.load_model(model_uri)

# Evaluate model
y_pred_clf = model.predict(X_test_clf)
print(classification_report(y_test_clf, y_pred_clf))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       0.62      0.62      0.62        13
           2       0.62      0.62      0.62        13

    accuracy                           0.78        45
   macro avg       0.74      0.74      0.74        45
weighted avg       0.78      0.78      0.78        45



---